# Data Cleaning — events tables

⚠️ **COST SAFETY**
- Every cell uses `maximum_bytes_billed = 10 GB` — if a query would scan more, it FAILS without charging.
- Default windows are short (7–14 days). Increase only after a successful dry run.
- If a cell fails with `Query exceeded limit for bytes billed`, narrow the window — do NOT raise the limit blindly.

In [ ]:
from google.cloud import bigquery
import pandas as pd

PROJECT = 'hopeful-list-429812-f3'
MAX_BYTES = 10 * 1024**3  # 10 GB hard cap per query

client = bigquery.Client(project=PROJECT)
job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES, use_query_cache=True)

def run(sql):
    return client.query(sql, job_config=job_config).to_dataframe()

## 1. Daily subscribe volume (last 14 days, ex-KZ)

In [ ]:
df_volume = run('''
SELECT DATE(timestamp) AS d,
       COUNT(*)              AS subs,
       COUNT(DISTINCT user_id) AS users
FROM `events.funnel-raw-table`
WHERE event_name = 'pr_funnel_subscribe'
  AND country_code != 'KZ'
  AND DATE(timestamp) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 14 DAY)
                          AND CURRENT_DATE()
GROUP BY d ORDER BY d
''')
df_volume.tail()

## 2. Top event names in funnel-raw-table (last 7 days)

In [ ]:
df_events = run('''
SELECT event_name, COUNT(*) AS n
FROM `events.funnel-raw-table`
WHERE timestamp BETWEEN TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
                   AND CURRENT_TIMESTAMP()
GROUP BY event_name ORDER BY n DESC LIMIT 25
''')
df_events

## 3. JSON parsing — quiz/path metadata in pr_funnel_subscribe (last 7 days)

In [ ]:
df_paths = run('''
SELECT
  COALESCE(
    JSON_VALUE(event_metadata, '$.scale_path'),
    JSON_VALUE(event_metadata, '$.escape_path'),
    JSON_VALUE(event_metadata, '$.starter_path'),
    JSON_VALUE(event_metadata, '$.simplify_path')
  ) AS path_from_meta,
  COUNT(*) AS subs
FROM `events.funnel-raw-table`
WHERE event_name = 'pr_funnel_subscribe'
  AND country_code != 'KZ'
  AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY path_from_meta ORDER BY subs DESC
''')
df_paths